In [1]:
import sys
print(sys.executable)

/venv/main/bin/python


In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Tue_May_27_02:21:03_PDT_2025
Cuda compilation tools, release 12.9, V12.9.86
Build cuda_12.9.r12.9/compiler.36037853_0


In [5]:
!/venv/main/bin/pip3 install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 31.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 43.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytorch_lightning]pytorch_lightning]


In [7]:
!/venv/main/bin/pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu129

Looking in indexes: https://download.pytorch.org/whl/cu129


In [1]:
from datasets import Dataset
from pycocotools.coco import COCO
from PIL import Image
import os
import transformers
from transformers import (
    DeformableDetrImageProcessor,
    DeformableDetrForObjectDetection,
    TrainingArguments,
    Trainer,
)
import torch
import warnings 

# ⚠️ Ignorar warnings irrelevantes do PyTorch
warnings.filterwarnings(
    "ignore",
    message=".*copying from a non-meta parameter.*",
    category=UserWarning,
    module="torch.nn.modules.module"
)

# ⚙️ Ativar autotuner da cuDNN (melhora desempenho em convoluções)
#torch.backends.cudnn.benchmark = True

# ============================================================
# 1️⃣ Caminhos do dataset COCO
# ============================================================
train_img_dir = "/workspace/SVRDD_COCO/train"
val_img_dir = "/workspace/SVRDD_COCO/valid"
test_img_dir = "/workspace/SVRDD_COCO/test"

train_path = f"{train_img_dir}/_annotations.coco.json"
val_path = f"{val_img_dir}/_annotations.coco.json"

# ============================================================
# 2️⃣ Labels
# ============================================================
id2label = {
    0: "longitudinal crack",
    1: "transverse crack",
    2: "alligator crack",
    3: "pothole",
    4: "manhole cover",
    5: "longitudinal patch",
    6: "transverse patch",
}
label2id = {v: k for k, v in id2label.items()}

# ============================================================
# 3️⃣ Conversão COCO → Dataset
# ============================================================
def coco_to_list(coco, image_dir):
    imgs = coco.loadImgs(coco.getImgIds())
    dataset = []
    for img in imgs:
        anns = coco.loadAnns(coco.getAnnIds(imgIds=img["id"]))
        objects = []
        for ann in anns:
            objects.append({
                "bbox": ann["bbox"],
                "category_id": ann["category_id"],
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0)
            })
        dataset.append({
            "image_id": img["id"],
            "image_path": os.path.join(image_dir, img["file_name"]),
            "objects": objects,
        })
    return dataset

print("Carregando dataset COCO...")
train_coco = COCO(train_path)
val_coco = COCO(val_path)

print("Convertendo para Dataset...")
train_dataset = Dataset.from_list(coco_to_list(train_coco, train_img_dir))
val_dataset = Dataset.from_list(coco_to_list(val_coco, val_img_dir))
print(f"Tamanhos: train={len(train_dataset)}, val={len(val_dataset)}")

# ============================================================
# 4️⃣ Modelo e Processor
# ============================================================
model_name = "SenseTime/deformable-detr"
processor = DeformableDetrImageProcessor.from_pretrained(model_name, do_convert_annotations=True)
model = DeformableDetrForObjectDetection.from_pretrained(
    model_name,
    num_labels=len(id2label),
    ignore_mismatched_sizes=True,
    id2label=id2label,
    label2id=label2id,
)

def collate_fn(batch):
    # O 'batch' aqui é uma lista de dicionários, onde cada dicionário
    # contém os dados brutos de uma imagem (image_path, objects, etc.)
    
    # 1. Extrair imagens e anotações do batch de dados brutos
    images = [Image.open(example["image_path"]).convert("RGB") for example in batch]
    
    # A estrutura de anotações deve ser no formato COCO (lista de dicionários)
    annotations = [
        {"image_id": example["image_id"], "annotations": example["objects"]}
        for example in batch
    ]
    
    # 2. Chamar o processor no BATCH COMPLETO.
    # O processor lida com:
    # a) Redimensionamento/Normalização.
    # b) Conversão das anotações COCO para o formato DETR (boxes normalizadas, classes).
    # c) PADDING e empilhamento dos tensores (pixel_values e pixel_mask).
    # d) Retorno dos tensores PyTorch prontos para o modelo.
    inputs = processor(images=images, annotations=annotations, return_tensors="pt")
    
    # inputs agora é um dicionário {pixel_values, pixel_mask, labels} prontos.
    return inputs


# ============================================================
# 6️⃣ Argumentos de treino — versão otimizada para 5090
# ============================================================
args = TrainingArguments(
    output_dir="./deformable-detr-finetuned-asphalt",
    per_device_train_batch_size=8,       # 🔼 DOBRADO
    per_device_eval_batch_size=2,
    learning_rate=1e-5,
    num_train_epochs=100,
    #weight_decay=1e-4,
    logging_dir="./logs",
    logging_steps=50,
    report_to="tensorboard",
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=3,
    remove_unused_columns=False,
    fp16=True,                           # ⚡ ATIVAR MIXED PRECISION
    gradient_accumulation_steps=1,       # Pode aumentar se quiser batch lógico maior
    # Paraleliza o carregamento
    torch_compile=False,                  # 🧠 Ativa compilação JIT do PyTorch 2.x (grande boost)
    dataloader_pin_memory=True,
    disable_tqdm=False,
    #warmup_steps=1000,
)

# ============================================================
# 7️⃣ Trainer
# ============================================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# ============================================================
# 8️⃣ Treinar!
# ============================================================
print("🚀 Iniciando treino otimizado na RTX 5090...")
trainer.train()


Carregando dataset COCO...
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Convertendo para Dataset...
Tamanhos: train=6000, val=1000


Some weights of the model checkpoint at SenseTime/deformable-detr were not used when initializing DeformableDetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DeformableDetrForObjectDetection wer

🚀 Iniciando treino otimizado na RTX 5090...


ValueError: matrix contains invalid numeric entries

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

import torch
import warnings
import os
from PIL import Image
from pycocotools.coco import COCO

# Importações do Hugging Face e PyTorch
from transformers import (
    DeformableDetrImageProcessor,
    DeformableDetrForObjectDetection,
    TrainingArguments,
    Trainer,
)
from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection # <--- Importação chave!

# ⚠️ Ignorar warnings (como os de precisão mista ou de compatibilidade)
warnings.filterwarnings(
    "ignore",
    message=".*copying from a non-meta parameter.*",
    category=UserWarning,
    module="torch.nn.modules.module"
)
warnings.filterwarnings(
    "ignore",
    message=".*The given NumPy array is not writeable.*",
    category=UserWarning,
    module="pycocotools.coco"
)


# ============================================================
# 1️⃣ Variáveis e Configuração do Ambiente
# ============================================================
# Definição do Device para a GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device detectado: {device}")

# Caminhos do dataset
ROOT_DIR = "/workspace/SVRDD_COCO"
TRAIN_DIRECTORY = os.path.join(ROOT_DIR, "train")
VAL_DIRECTORY = os.path.join(ROOT_DIR, "valid")
ANNOTATION_FILE_NAME = "_annotations.coco.json" # Nome padrão do seu arquivo

# Labels
id2label = {
    0: "longitudinal crack", 1: "transverse crack", 2: "alligator crack",
    3: "pothole", 4: "manhole cover", 5: "longitudinal patch",
    6: "transverse patch",
}
label2id = {v: k for k, v in id2label.items()}

# Modelo base
model_name = "SenseTime/deformable-detr"


# ============================================================
# 2️⃣ Classe Customizada para Dataloading Robusto
# ============================================================
class CocoDetectionCustom(CocoDetection):
    # Herda a lógica de leitura de arquivos COCO do torchvision
    def __init__(self, image_directory_path: str, image_processor):
        annotation_file_path = os.path.join(image_directory_path, ANNOTATION_FILE_NAME)
        
        # O __init__ do CocoDetection carrega as anotações e os caminhos
        super(CocoDetectionCustom, self).__init__(image_directory_path, annotation_file_path)
        
        self.image_processor = image_processor

    def __getitem__(self, idx):
        # 1. Obter Imagem (PIL Image) e Anotações (lista COCO)
        image, annotations = super(CocoDetectionCustom, self).__getitem__(idx)
        image_id = self.ids[idx] # Obtém o ID da imagem do índice
        
        
        # 2. Formatar anotações para o processor do Hugging Face
        annotations = {'image_id': image_id, 'annotations': annotations}
        
        # 3. Processar a amostra (redimensionamento e conversão COCO -> DETR)
        # return_tensors="pt" é aplicado aqui, mas sem padding
        encoding = self.image_processor(images=image, annotations=annotations, return_tensors="pt")
        
        # 4. Retornar amostra sem a dimensão de batch (squeeze)
        return {
            "pixel_values": encoding["pixel_values"].squeeze(),
            "pixel_mask": encoding["pixel_mask"].squeeze(),
            "labels": encoding["labels"][0], # O labels é uma lista, pegamos o primeiro elemento
        }

# ============================================================
# 3️⃣ Inicialização do Modelo e Datasets
# ============================================================
print("Carregando modelo e processor...")
processor = DeformableDetrImageProcessor.from_pretrained(model_name)

print("Instanciando Datasets...")
TRAIN_DATASET = CocoDetectionCustom(
    image_directory_path=TRAIN_DIRECTORY, 
    image_processor=processor
)
VAL_DATASET = CocoDetectionCustom(
    image_directory_path=VAL_DIRECTORY, 
    image_processor=processor
)
TEST_DATASET = CocoDetectionCustom(
    image_directory_path=VAL_DIRECTORY, 
    image_processor=processor
)

# ============================================================
# 4️⃣ Collate Function para BATCHING e PADDING
# ============================================================
def collate_fn(batch):
    pixel_values = [item["pixel_values"] for item in batch]
    encoding = processor.pad(pixel_values, return_tensors="pt")
    labels = [item["labels"] for item in batch]
    return {
        'pixel_values': encoding['pixel_values'],
        'pixel_mask': encoding['pixel_mask'],
        'labels': labels
    }

num_workers = min(8, max(1, (os.cpu_count() or 4)//2))
print("num_workers =", num_workers)

params_data_loader = {
    "collate_fn": collate_fn, 
    "batch_size": 8, 
    "num_workers": num_workers,
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 2
}

TRAIN_DATALOADER = DataLoader(dataset=TRAIN_DATASET, shuffle=True, **params_data_loader)
VAL_DATALOADER = DataLoader(dataset=VAL_DATASET, shuffle=False, **params_data_loader)
TEST_DATALOADER = DataLoader(dataset=TEST_DATASET, **params_data_loader)

Device detectado: cuda:0
Carregando modelo e processor...
Instanciando Datasets...
loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
num_workers = 8


In [1]:
from transformers import DeformableDetrForObjectDetection
import pytorch_lightning as pl
import torch.nn as nn

class DeformableDetr(pl.LightningModule): # Mude o nome da classe para clareza
    def __init__(self, lr, lr_backbone, weight_decay):
        super().__init__()
        
        self.model = DeformableDetrForObjectDetection.from_pretrained(
            pretrained_model_name_or_path=model_name, 
            num_labels=len(id2label),
            ignore_mismatched_sizes=True
        )
        self.reinit_detection_heads()
        self.lr = lr
        self.lr_backbone = lr_backbone
        self.weight_decay = weight_decay

    def reinit_detection_heads(self):
        for name, module in self.model.named_modules():
            if "class_embed" in name or "bbox_embed" in name:
                for pn, p in module.named_parameters(recurse=False):
                    with torch.no_grad():
                        if p.dim() > 1:
                            nn.init.xavier_uniform_(p)
                        else:
                            p.zero_()
        print("Heads (class_embed / bbox_embed) reinicializadas.")
    
    def forward(self, pixel_values, pixel_mask):
        return self.model(pixel_values=pixel_values, pixel_mask=pixel_mask)

    def common_step(self, batch, batch_idx):
        # Move explicitamente para o device
        pixel_values = batch["pixel_values"].to(self.device)
        pixel_mask = batch["pixel_mask"].to(self.device)
        labels = [{k: v.to(self.device) for k, v in t.items()} for t in batch["labels"]]
    
        outputs = self.model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
    
        loss = outputs.loss
        loss_dict = outputs.loss_dict
        return loss, loss_dict

    def training_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)     
        # logs metrics for each training_step, and the average across the epoch
        self.log("training_loss", loss)
        for k,v in loss_dict.items():
            self.log("train_" + k, v.item())

        return loss

    def validation_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)     
        self.log("validation/loss", loss)
        for k, v in loss_dict.items():
            self.log("validation_" + k, v.item())
            
        return loss

    def configure_optimizers(self):
        # DETR authors decided to use different learning rate for backbone
        # you can learn more about it here: 
        # - https://github.com/facebookresearch/detr/blob/3af9fa878e73b6894ce3596450a8d9b89d918ca9/main.py#L22-L23
        # - https://github.com/facebookresearch/detr/blob/3af9fa878e73b6894ce3596450a8d9b89d918ca9/main.py#L131-L139
        param_dicts = [
            {
                "params": [p for n, p in self.named_parameters() if "backbone" not in n and p.requires_grad]},
            {
                "params": [p for n, p in self.named_parameters() if "backbone" in n and p.requires_grad],
                "lr": self.lr_backbone,
            },
        ]
        return torch.optim.AdamW(param_dicts, lr=self.lr, weight_decay=self.weight_decay)

    def train_dataloader(self):
        return TRAIN_DATALOADER

    def val_dataloader(self):
        return VAL_DATALOADER

ModuleNotFoundError: No module named 'transformers'

In [3]:
# ============================================================
# 5️⃣ TrainingArguments (Estável e Otimizado)
# ============================================================

model = DeformableDetr(lr=1e-4, lr_backbone=1e-5, weight_decay=1e-4)

batch = next(iter(TRAIN_DATALOADER))
outputs = model(pixel_values=batch['pixel_values'], pixel_mask=batch['pixel_mask'])

Some weights of the model checkpoint at SenseTime/deformable-detr were not used when initializing DeformableDetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DeformableDetrForObjectDetection wer

Heads (class_embed / bbox_embed) reinicializadas.


In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

#args = TrainingArguments(
#    output_dir="./deformable-detr-finetuned-asphalt",
#    per_device_train_batch_size=8,   # Uso eficiente da 5090
#    per_device_eval_batch_size=4,
#    gradient_accumulation_steps=1,   
#    learning_rate=1e-5,              # 🚨 LR de segurança (aumentar após estabilidade)
#    warmup_steps=1000,               # ✅ Essencial para estabilidade
#    num_train_epochs=50,             # Número razoável de épocas
#    logging_dir="./logs",
#    logging_steps=100,
#    report_to="tensorboard",
#    disable_tqdm=False,
#    save_strategy="epoch",
#    eval_strategy="epoch",
#    fp16=False,                      # 🛑 FP32 para máxima estabilidade inicial
#    remove_unused_columns=False,
#    dataloader_pin_memory=True,      # Otimização de I/O
#)

# ============================================================
# 6️⃣ Trainer e Início
# ============================================================

checkpoint = ModelCheckpoint(
    monitor="validation/loss",
    mode="min",
    save_top_k=1,
    filename="best-{epoch}-{val_loss:.2f}"
)

early_stop = EarlyStopping(
    monitor="validation/loss",
    patience=12,
    mode="min"
)

trainer = Trainer(
    devices=1, 
    accelerator="gpu", 
    max_epochs=100, 
    gradient_clip_val=0.1, 
    accumulate_grad_batches=4, 
    log_every_n_steps=100, 
    precision=16,
    callbacks=[checkpoint, early_stop],
)


print("🚀 Iniciando treinamento com Pytorch Trainer...")
trainer.fit(model)

/venv/main/lib/python3.12/site-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 5090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


🚀 Iniciando treinamento com Pytorch Trainer...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/venv/main/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type                             | Params | Mode
------------------------------------------------------------------
0 | model | DeformableDetrForObjectDetection | 40.0 M | eval
------------------------------------------------------------------
39.8 M    Trainable params
222 K     Non-trainable params
40.0 M    Total params
160.192   Total estimated model params size (MB)
0         Modules in train mode
425       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:484: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
/venv/main/lib/python3.12/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/venv/main/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:527: Found 425 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
/home/andre/TCC/models/VitDet/output_vitdet/lightning_logs/version_12/checkpoints/best-epoch=92-val_loss=0.00.ckpt